# BERTopic clustering with BGE-M3, UMAP, and HDBSCAN

This notebook mirrors the aggregation pipeline pattern used in the other notebooks, but uses `BAAI/bge-m3` embeddings inside a BERTopic workflow. UMAP reduces the embedding space, HDBSCAN discovers dense event/topic clusters, and `CountVectorizer` uses a trilingual stopword list to suppress boilerplate vocabulary while preserving meaningful Sinhala, Tamil, and English terms.

In [1]:
# Install dependencies when running in a fresh notebook environment.
# Restart the kernel if any package is newly installed.
!pip install -q bertopic sentence-transformers umap-learn hdbscan grapheme
!pip install -U "scipy>=1.15.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 4.6 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 1.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 52.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.0 which is incompatible.


In [2]:
from pathlib import Path
import os
import re
import unicodedata

import numpy as np
import pandas as pd
import grapheme

SEED = 42
np.random.seed(SEED)

NOTEBOOK_DIR = Path.cwd()

# Dataset location
DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")

# Writable location for results
RESULTS_ROOT = Path("/kaggle/working/results")
RESULTS_DIR = RESULTS_ROOT / "bge_m3_bertopic"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/bge_m3_bertopic


In [3]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [4]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [5]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [6]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [7]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
print(df["text"][10])

2025.12.07 වන දින රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් බිඳ රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් (57,268,000/-) වටිනා ප්‍රාඩෝ වර්ගයේ ජීප් රථයක්, රන් භාණ්ඩ මුදල් සහ ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට ලද පැමිණිල්ලක් මත විමර්ශන ආරම්භ කර තිබුණි.

2025.12.14 වන දින ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල, මාතවතගම හා විල්ගමුව ප්‍රදේශවලදී හෙරොයින්  ග්‍රෑම් 10 මිලිග්‍රෑම් 300 ක්, අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් 10 මිලිග්‍රෑම් 250 ක්, අපරාධයට යොදාගත් අත්වැසුම් ජෝඩු 04 ක්, උල් පිහි 04 ක්, අපරාධයට යොදා ගත් යතුරුපැදියක්, වෑන් රථයක් හා ලොරි රථයක් සමඟ අත්අඩංගුවට ගෙන ඇත.

සැකකරුවන් අවුරුදු 34, 36, 38, 40, 43 හා 48 වයස්වල පසුවන කුරුණෑගල, නුගගොල්ල, මාවතගම, මැල්සිරිපුර හා මහනුවර ප්‍රදේශවල පදිංචිකරුවන් වේ.

සැකකරුවන් විසින් කොල්ලකන ලද ප්‍රාඩෝ වර්ගයේ ජීප් රථය මීගලෑව ප්‍රදේශයේ දමා ගොස් තිබියදී සොයාගෙන ඇත. තවද කොල්ලකන ලද මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක (5,939,000/-) මුදලක් හා කොල්ලකන ලද මුදල් වලින් උපයාගන්නා ලද දේපළ 

In [9]:
import unicodedata

def clean_text(text):
    # Replace punctuation with spaces
    text = "".join(
        " " if unicodedata.category(char).startswith("P") else char
        for char in text
    )

    # Keep words with more than 2 graphemes
    return " ".join(
        word for word in text.split()
        if len(list(grapheme.graphemes(word))) > 2
    )

df["text"] = df["text"].apply(clean_text)

In [10]:
print(df["text"][10])

2025 රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්‍රාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්‍රදේශවලදී හෙරොයින් ග්‍රෑම් මිලිග්‍රෑම් 300 අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් මිලිග්‍රෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්‍රදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්‍රාඩෝ වර්ගයේ රථය මීගලෑව ප්‍රදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව මහේස්ත්‍රාත් අධිකරණයට ඉදිරිපත් වැඩිදුර විමර්ශන සඳහා 2025 දක්වා රැඳවුම් නියෝග ලබාගෙන ගල්නෑව පොලිසිය වැඩිදුර විමර්ශන කරනු ලබයි


In [11]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()
print(df["text"][10])

df shape after text cleaning: (1999, 7)
2025 රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්‍රාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්‍රදේශවලදී හෙරොයින් ග්‍රෑම් මිලිග්‍රෑම් 300 අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් මිලිග්‍රෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්‍රදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්‍රාඩෝ වර්ගයේ රථය මීගලෑව ප්‍රදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව මහේස්ත්‍රාත් අධිකරණයට ඉදිරිපත් වැඩිදුර විමර්ශන සඳහා 2025 දක්වා රැඳවුම් නියෝග ලබාගෙන ගල්නෑව පොලිසිය වැඩිදුර විමර්ශන කරනු ලබ

In [12]:
# Build documents for semantic embedding and topic representation
def build_document(title, body):
    title = str(title).strip()
    body = str(body).strip()
    return f"{title}. {body}" if title else body


df["document"] = df.apply(lambda row: build_document(row["title"], row["text"]), axis=1)
documents = df["document"].tolist()

print(f"Documents: {len(documents)}")
documents[0][:500]
print(df["text"][10])

Documents: 1999
2025 රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්‍රාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්‍රදේශවලදී හෙරොයින් ග්‍රෑම් මිලිග්‍රෑම් 300 අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් මිලිග්‍රෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්‍රදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්‍රාඩෝ වර්ගයේ රථය මීගලෑව ප්‍රදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව මහේස්ත්‍රාත් අධිකරණයට ඉදිරිපත් වැඩිදුර විමර්ශන සඳහා 2025 දක්වා රැඳවුම් නියෝග ලබාගෙන ගල්නෑව පොලිසිය වැඩිදුර විමර්ශන කරනු ලබයි


In [13]:
print("Sample document:", documents[10:50])

Sample document: ['බුල්නෑව නිවසක් බිඳ ජීප් රථයක්, රන් භාණ්ඩ හා මුදල් කෝල්ල කෑ සැකකරුවන් අල්ලයි. 2025 රාත්\u200dරී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්\u200dරදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්\u200dරාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්\u200dරදේශවලදී හෙරොයින් ග්\u200dරෑම් මිලිග්\u200dරෑම් 300 අයිස් මත්ද්\u200dරව්\u200dය ග්\u200dරෑම් මිලිග්\u200dරෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්\u200dරදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්\u200dරාඩෝ වර්ගයේ රථය මීගලෑව ප්\u200dරදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව ම

In [14]:


stopwords = {
    "අද", "අප", "අපි", "අපේ", "අතර", "අනුව", "අය", "අයගේ", "ඇත", "ඇති", "ඇතුළු",
    "ඒ", "එම", "එය", "එහි", "එක්", "එක", "එකක්", "කර", "කරන", "කරයි", "කරමින්",
    "කළ", "කිරීම", "කියන", "කියා", "ගැන", "ගැනීමට", "ගැනීම", "තම", "තුළ", "ද", "දී",
    "දක්වා", "දැයි", "නම්", "නමුත්", "නිසා", "බව", "බවට", "බවයි", "බවද", "මත", "මෙන්",
    "මෙම", "මේ", "ය", "යන", "ලෙස", "වූ", "වෙත", "වෙනුවෙන්", "සහ", "සිට", "සඳහා",
    "හා", "හෝ", "මගින්", "බවත්"
}



stopwords = sorted(stopwords)

print(f"Stopwords: {len(stopwords)}")

Stopwords: 59


In [15]:
# Embed documents with BGE-M3
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "BAAI/bge-m3"
embedding_model = SentenceTransformer(embedding_model_name, device=device)

embeddings = embedding_model.encode(
    documents,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

(1999, 1024)


In [16]:
# Embedding separation score (comparable across notebooks): 1 - mean pairwise cosine similarity
# on a random 100-document sample of the raw (pre-UMAP) embeddings.
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sample_emb = embeddings[sample_idx]

sims = cosine_similarity(sample_emb)
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()

print(f"Embedding model: {embedding_model_name}")
print(f"mean_sim={pairwise.mean():.4f} std={pairwise.std():.4f} min={pairwise.min():.4f} max={pairwise.max():.4f}")
print(f"Separation score (1 - mean cosine similarity): {separation_score:.4f}")

Embedding model: BAAI/bge-m3
mean_sim=0.4053 std=0.0668 min=0.2252 max=0.7894
Separation score (1 - mean cosine similarity): 0.5947


In [17]:
# Configure BERTopic with UMAP, HDBSCAN, and CountVectorizer
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

umap_model = UMAP(
    n_neighbors=3,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words=list(stopwords),
    token_pattern=r"(?u)\b\w\w+\b",
    ngram_range=(1, 2),
    min_df=1,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

In [18]:
# Fit BERTopic using the precomputed BGE-M3 embeddings
topics, probabilities = topic_model.fit_transform(documents, embeddings)

df["topic_id"] = topics
df["topic_probability"] = [float(np.max(row)) if row is not None and len(row) else np.nan for row in probabilities]

topic_info = topic_model.get_topic_info()
topic_info.head(20)

2026-08-20 12:05:45,756 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-08-20 12:06:04,665 - BERTopic - Dimensionality - Completed ✓
2026-08-20 12:06:04,666 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-20 12:06:19,580 - BERTopic - Cluster - Completed ✓
2026-08-20 12:06:19,589 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-20 12:06:21,585 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,160,-1_මන රම_පත_ජන පත_ජන,"[මන රම, පත, ජන පත, ජන, සම හය, ගණක පත, පත වරය, ...",[LOLC සමූහය ශක්තිමත් වර්ධනයක් වාර්තා කරයි. 202...
1,0,11,0_මහජන උපය_උපය සම_උපය_බල මණ,"[මහජන උපය, උපය සම, උපය, බල මණ, මහජන, මණ ඩලය, ඩ...",[විදුලි ගාස්තු ඉහළ දැමීමේ තීරණය 15වැනිදා. විදු...
2,1,8,1_නව කම_කම උල_උල_ලධ,"[නව කම, කම උල, උල, ලධ, උල ඝනය, ඝනය, කම සම, සත ...",[මානව හිමිකම් කොමිසම සිය මානව හිමිකම් කඩකළ බවට...
3,2,8,2_තරග තහනම_ෂක ජයත_ලක මහත_පර රව,"[තරග තහනම, ෂක ජයත, ලක මහත, පර රව, රව රම, අඩ වට...",[ශිරාන් බාසික් නිදොස් කොට නිදහස්. ප්‍රධාන මත්ද...
4,3,7,3_අයව_රත දන_අධ පන_පන රත,"[අයව, රත දන, අධ පන, පන රත, බඳවද, අයව රත, 2026,...",[සියලු මන්ත්‍රීවරුන්ට අධ්‍යාපන ප්‍රතිසංස්කරණ ග...
5,4,7,4_අග_අග යවර_අග ආච_ආච හර,"[අග, අග යවර, අග ආච, ආච හර, යථ තත, රව යමය, තත ව...",[ජීවිත ගොඩනගන්න නිලධාරීන්ගේ කාර්යක්ෂම සහය අවශ්...
6,5,7,5_කම වන_කම_චය සභ_අර ධක,"[කම වන, කම, චය සභ, අර ධක, ධක අරම, 200ක, රජය, ව...",[වතු කම්කරුවන් ගැන කම්කරු විනිශ්චය සභාව දුන් න...
7,6,7,6_ටලය_හග_රව රම_හග නය,"[ටලය, හග, රව රම, හග නය, රව, රව සමඟ, ළඹ මධ, සන ...",[කෝණා සමග කුඩු සැඟවූ නෝනලා. පොලීසියට පෙන්වා පළ...
8,7,7,7_දමර_ඇළට_සය දමර_ටර රථය,"[දමර, ඇළට, සය දමර, ටර රථය, අනත රක, ගමන කරම, ටර...",[බිරිඳගේ හිස රැගෙන පොලීසියට ගිය අනිල්. වනවිට ස...
9,8,7,8_රම මල_මල_පර ෂකවරය_ෂකවරය,"[රම මල, මල, පර ෂකවරය, ෂකවරය, නවත, මල මහ, රස ලධ...",[රිමාන්ඩ් - ජනමතය නිවැරැදිද?. කිසියම් චෝදනාවකට...


In [19]:
# Clustering evaluation: unsupervised metrics on the UMAP-reduced space HDBSCAN actually clustered on
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

reduced_embeddings = topic_model.umap_model.embedding_

mask = df["topic_id"] != -1
X_valid = reduced_embeddings[mask.values]
labels_valid = df.loc[mask, "topic_id"].values
n_clusters = len(set(labels_valid))
noise_ratio = 1 - mask.mean()

print(f"Model: {embedding_model_name} + UMAP + HDBSCAN (BERTopic)")
print(f"Articles: {len(df)} | Topics (excl. noise): {n_clusters} | Noise ratio: {noise_ratio:.2%}")

if n_clusters > 1:
    sil = silhouette_score(X_valid, labels_valid, metric="cosine")
    dbi = davies_bouldin_score(X_valid, labels_valid)
    ch = calinski_harabasz_score(X_valid, labels_valid)
    print(f"Silhouette Score (cosine): {sil:.4f}")
    print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
    print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")
else:
    sil = dbi = ch = float("nan")
    print("Not enough clusters to compute silhouette/DBI/CH.")

# Save scores so they can be compared against the other embedding models later
scores_path = RESULTS_DIR / "bge_m3_bertopic_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "UMAP+HDBSCAN (BERTopic)",
    "embedding_dim": embeddings.shape[1],
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": round(noise_ratio, 4),
    "silhouette": round(sil, 4) if n_clusters > 1 else None,
    "davies_bouldin": round(dbi, 4) if n_clusters > 1 else None,
    "calinski_harabasz": round(ch, 2) if n_clusters > 1 else None,
}
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

Model: BAAI/bge-m3 + UMAP + HDBSCAN (BERTopic)
Articles: 1999 | Topics (excl. noise): 613 | Noise ratio: 8.00%
Silhouette Score (cosine): 0.8060
Davies-Bouldin Index:      0.3604  (lower is better)
Calinski-Harabasz Index:   25719.44  (higher is better)
Saved scores to /kaggle/working/results/bge_m3_bertopic/bge_m3_bertopic_scores.csv


In [20]:
# Cluster/topic summary
print(df["topic_id"].value_counts().head(20))
print("Noise articles:", int((df["topic_id"] == -1).sum()))
print("Number of topics found:", df["topic_id"].nunique() - (1 if -1 in df["topic_id"].values else 0))

topic_info.head(30)

topic_id
-1     160
 0      11
 1       8
 2       8
 3       7
 4       7
 5       7
 6       7
 7       7
 8       7
 9       7
 10      7
 11      7
 12      6
 13      6
 14      6
 15      6
 16      6
 17      6
 18      6
Name: count, dtype: int64
Noise articles: 160
Number of topics found: 613


,Topic,Count,Name,Representation,Representative_Docs
0,-1,160,-1_මන රම_පත_ජන පත_ජන,"[මන රම, පත, ජන පත, ජන, සම හය, ගණක පත, පත වරය, ...",[LOLC සමූහය ශක්තිමත් වර්ධනයක් වාර්තා කරයි. 202...
1,0,11,0_මහජන උපය_උපය සම_උපය_බල මණ,"[මහජන උපය, උපය සම, උපය, බල මණ, මහජන, මණ ඩලය, ඩ...",[විදුලි ගාස්තු ඉහළ දැමීමේ තීරණය 15වැනිදා. විදු...
2,1,8,1_නව කම_කම උල_උල_ලධ,"[නව කම, කම උල, උල, ලධ, උල ඝනය, ඝනය, කම සම, සත ...",[මානව හිමිකම් කොමිසම සිය මානව හිමිකම් කඩකළ බවට...
3,2,8,2_තරග තහනම_ෂක ජයත_ලක මහත_පර රව,"[තරග තහනම, ෂක ජයත, ලක මහත, පර රව, රව රම, අඩ වට...",[ශිරාන් බාසික් නිදොස් කොට නිදහස්. ප්‍රධාන මත්ද...
4,3,7,3_අයව_රත දන_අධ පන_පන රත,"[අයව, රත දන, අධ පන, පන රත, බඳවද, අයව රත, 2026,...",[සියලු මන්ත්‍රීවරුන්ට අධ්‍යාපන ප්‍රතිසංස්කරණ ග...
5,4,7,4_අග_අග යවර_අග ආච_ආච හර,"[අග, අග යවර, අග ආච, ආච හර, යථ තත, රව යමය, තත ව...",[ජීවිත ගොඩනගන්න නිලධාරීන්ගේ කාර්යක්ෂම සහය අවශ්...
6,5,7,5_කම වන_කම_චය සභ_අර ධක,"[කම වන, කම, චය සභ, අර ධක, ධක අරම, 200ක, රජය, ව...",[වතු කම්කරුවන් ගැන කම්කරු විනිශ්චය සභාව දුන් න...
7,6,7,6_ටලය_හග_රව රම_හග නය,"[ටලය, හග, රව රම, හග නය, රව, රව සමඟ, ළඹ මධ, සන ...",[කෝණා සමග කුඩු සැඟවූ නෝනලා. පොලීසියට පෙන්වා පළ...
8,7,7,7_දමර_ඇළට_සය දමර_ටර රථය,"[දමර, ඇළට, සය දමර, ටර රථය, අනත රක, ගමන කරම, ටර...",[බිරිඳගේ හිස රැගෙන පොලීසියට ගිය අනිල්. වනවිට ස...
9,8,7,8_රම මල_මල_පර ෂකවරය_ෂකවරය,"[රම මල, මල, පර ෂකවරය, ෂකවරය, නවත, මල මහ, රස ලධ...",[රිමාන්ඩ් - ජනමතය නිවැරැදිද?. කිසියම් චෝදනාවකට...


In [21]:
# Inspect sample titles per topic
for topic_id, group in df[df["topic_id"] != -1].groupby("topic_id"):
    print(f"=== Topic {topic_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

    

=== Topic 0 (11 articles) ===
- විදුලි බිල සියයට 30කින් අඩු කළ හැකියි
- බස් ගාස්තු පහළට
- බිල්පත් ගෙවීම් ගැන විදුලි පාරිභෝගිකයින්ට දැනුම්දීමක්
- විදුලි බල මණ්ඩලය ඉල්ලා සිටි 6.8%ක විදුලි ගාස්තු වැඩි කිරීම මහජන උපයෝගීතා කොමිසම ප්‍රතික්ෂේප කළේ ඇයි?
- වසර තුනකින් විදුලි ගාස්තු තවත් පහළට
- විදුලි ගාස්තු ඉහළ දැමීමේ තීරණය 15වැනිදා
- යෝජිත විදුලි ගාස්තු සංශෝධනය ගැන මහජන අදහස් ලබාගැනීම අද
- ලංවිම ප්‍රතිසංස්කරණ ඉදිරියට, පෞද්ලීකරණය නවතී
- විදුලි ගාස්තු සංශෝදනය ගැන සාවද්‍ය වාර්තා මණ්ඩලය ප්‍රතික්ෂේප කරයි
- සියයට 13.5ක විදුලි ගාස්තු වැඩිවීමක් ඉල්ලයි

=== Topic 1 (8 articles) ===
- ශ්‍රී ලංකා පොලිසියේ මානව හිම්කම් උල්ලංඝණ ලැයිස්තුව තවත් දිගු වෙයි
- පොලිස් නිලධාරියකු සිවිල් ඇඳුමින් කරන ක්‍රියාවන්හි වගකීමෙන් රජයට බැහැර විය නොහැකි බවට ශ්‍රේෂ්ඨාධිකරණයෙන් තීන්දුවක්
- මානව හිමිකම් කොමිසම සිය මානව හිමිකම් කඩකළ බවට එහි නිලධාරිනියක් අඛණ්ඩ සත්‍යග්‍රහයක නිරත වූයේ ඇයි?
- ශ්‍රී ලංකා පොලිසියේ මානව හිමිකම් උල්ලංඝන: වින්දිතයින්ට කෝටි 15ක් වන්දි ගෙවන්නට සිදු වූ පොලිස් නිලධාරීන් කවුද?
- රත්නපුර හිටපු දිසාපතිවරිය, පොලි

In [22]:
# Save article-level assignments and topic summaries
assignments_path = RESULTS_DIR / "bge_m3_bertopic_assignments.csv"
topics_path = RESULTS_DIR / "bge_m3_bertopic_topics.csv"

df.drop(columns=["document"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
topic_info.to_csv(topics_path, index=False, encoding="utf-8-sig")

print(f"Saved assignments: {assignments_path.resolve()}")
print(f"Saved topics: {topics_path.resolve()}")

Saved assignments: /kaggle/working/results/bge_m3_bertopic/bge_m3_bertopic_assignments.csv
Saved topics: /kaggle/working/results/bge_m3_bertopic/bge_m3_bertopic_topics.csv


In [23]:
# Optional BERTopic visualizations
# These are useful in Jupyter/Colab but can be slow for large corpora.

# topic_model.visualize_topics()
# topic_model.visualize_barchart(top_n_topics=20)
# topic_model.visualize_hierarchy()